In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import grangercausalitytests
from scipy.stats import spearmanr
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.drawing.image import Image
from io import BytesIO

# Load the data from the Excel file
file_path = r"C:\Users\rohit\Desktop\research paper\granger.xlsx"
data = pd.read_excel(file_path)

# Extract the relevant columns
vix = data['VIX']
srisk = data['SRISK']
sentiment_index = data['sentiment index']

# Granger Causality Test
def granger_causality_test(df, max_lag=5):
    results = {}
    for lag in range(1, max_lag + 1):
        test_result = grangercausalitytests(df, max_lag, addconst=True, verbose=False)
        results[lag] = test_result[lag][0]['ssr_chi2test'][1]  # p-value
    return results

# Perform Granger causality tests
results_vix_sentiment = granger_causality_test(pd.DataFrame({'VIX': vix, 'sentiment index': sentiment_index}))
results_sentiment_vix = granger_causality_test(pd.DataFrame({'sentiment index': sentiment_index, 'VIX': vix}))

results_srisk_sentiment = granger_causality_test(pd.DataFrame({'SRISK': srisk, 'sentiment index': sentiment_index}))
results_sentiment_srisk = granger_causality_test(pd.DataFrame({'sentiment index': sentiment_index, 'SRISK': srisk}))

# Non-parametric test (Spearman correlation)
def spearman_correlation(df):
    return spearmanr(df.iloc[:, 0], df.iloc[:, 1]).correlation

# Spearman correlation
spearman_vix_sentiment = spearman_correlation(pd.DataFrame({'VIX': vix, 'sentiment index': sentiment_index}))
spearman_sentiment_vix = spearman_correlation(pd.DataFrame({'sentiment index': sentiment_index, 'VIX': vix}))

spearman_srisk_sentiment = spearman_correlation(pd.DataFrame({'SRISK': srisk, 'sentiment index': sentiment_index}))
spearman_sentiment_srisk = spearman_correlation(pd.DataFrame({'sentiment index': sentiment_index, 'SRISK': srisk}))

# Interpret results based on a significance level of 0.05
def interpret_results(results):
    interpretation = {}
    for lag, p_value in results.items():
        interpretation[lag] = 'Reject Null Hypothesis' if p_value < 0.05 else 'Fail to Reject Null Hypothesis'
    return interpretation

# Interpret results
causality_vix_sentiment = interpret_results(results_vix_sentiment)
causality_sentiment_vix = interpret_results(results_sentiment_vix)

causality_srisk_sentiment = interpret_results(results_srisk_sentiment)
causality_sentiment_srisk = interpret_results(results_sentiment_srisk)

# Format results for DataFrame
def format_granger_results(results, causality):
    lags = list(results.keys())
    p_values = list(results.values())
    causality_status = [causality.get(lag, 'Not Tested') for lag in lags]
    return lags, p_values, causality_status

lags_vix_sentiment, p_values_vix_sentiment, causality_vix_sentiment = format_granger_results(results_vix_sentiment, causality_vix_sentiment)
lags_sentiment_vix, p_values_sentiment_vix, causality_sentiment_vix = format_granger_results(results_sentiment_vix, causality_sentiment_vix)

lags_srisk_sentiment, p_values_srisk_sentiment, causality_srisk_sentiment = format_granger_results(results_srisk_sentiment, causality_srisk_sentiment)
lags_sentiment_srisk, p_values_sentiment_srisk, causality_sentiment_srisk = format_granger_results(results_sentiment_srisk, causality_sentiment_srisk)

# Create plots
def plot_granger_results(lags, p_values, title, ax):
    ax.plot(lags, p_values, marker='o', linestyle='-', color='b')
    ax.axhline(y=0.05, color='r', linestyle='--')
    ax.set_xlabel('Lag')
    ax.set_ylabel('p-value')
    ax.set_title(title)
    ax.grid(True)

# Plot and save graphs
def save_plot():
    fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    plot_granger_results(lags_vix_sentiment, p_values_vix_sentiment, 'VIX → Sentiment', axs[0, 0])
    plot_granger_results(lags_sentiment_vix, p_values_sentiment_vix, 'Sentiment → VIX', axs[0, 1])
    plot_granger_results(lags_srisk_sentiment, p_values_srisk_sentiment, 'SRISK → Sentiment', axs[1, 0])
    plot_granger_results(lags_sentiment_srisk, p_values_sentiment_srisk, 'Sentiment → SRISK', axs[1, 1])
    plt.tight_layout()

    # Save plot to BytesIO
    img_stream = BytesIO()
    plt.savefig(img_stream, format='png')
    img_stream.seek(0)
    plt.close()
    return img_stream

# Save to Excel
wb = Workbook()
ws = wb.active
ws.title = "Granger Causality Results"

# Add tables
results_vix_sentiment_df = pd.DataFrame({
    'Lag': lags_vix_sentiment,
    'VIX → Sentiment - p-value': p_values_vix_sentiment,
    'VIX → Sentiment - Causality': causality_vix_sentiment
})

results_sentiment_vix_df = pd.DataFrame({
    'Lag': lags_sentiment_vix,
    'Sentiment → VIX - p-value': p_values_sentiment_vix,
    'Sentiment → VIX - Causality': causality_sentiment_vix
})

results_srisk_sentiment_df = pd.DataFrame({
    'Lag': lags_srisk_sentiment,
    'SRISK → Sentiment - p-value': p_values_srisk_sentiment,
    'SRISK → Sentiment - Causality': causality_srisk_sentiment
})

results_sentiment_srisk_df = pd.DataFrame({
    'Lag': lags_sentiment_srisk,
    'Sentiment → SRISK - p-value': p_values_sentiment_srisk,
    'Sentiment → SRISK - Causality': causality_sentiment_srisk
})

# Write tables to Excel
for r in dataframe_to_rows(results_vix_sentiment_df, index=False, header=True):
    ws.append(r)
ws.append([])
for r in dataframe_to_rows(results_sentiment_vix_df, index=False, header=True):
    ws.append(r)
ws.append([])
for r in dataframe_to_rows(results_srisk_sentiment_df, index=False, header=True):
    ws.append(r)
ws.append([])
for r in dataframe_to_rows(results_sentiment_srisk_df, index=False, header=True):
    ws.append(r)

# Save plot
img_stream = save_plot()
img = Image(img_stream)
ws.add_image(img, 'A20')

# Save Excel file
excel_file_path = r'C:\Users\rohit\Desktop\granger_analysis_results.xlsx'
wb.save(excel_file_path)

print(f"Granger causality analysis results and plots have been saved to '{excel_file_path}'.")


C:\Python311\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger causality analysis results and plots have been saved to 'C:\Users\rohit\Desktop\granger_analysis_results.xlsx'.
